In [36]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


df=pd.read_csv('data/Churn_Modelling_cleaned.csv')
df.head(5)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,BalanceToSalaryRatio,IsHighBalance,IsMultiProductCustomer,AgeGroup,CreditScoreBand,TenurePerAge,ActiveMember_And_MultiProduct
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,0.000000,0,0,36-45,Good,0.047619,0
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,0.744677,0,0,36-45,Good,0.024390,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1.401375,1,1,36-45,Fair,0.190476,0
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0.000000,0,1,36-45,Good,0.025641,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,1.587055,1,0,36-45,Excellent,0.046512,0


## Data Preprocessing and Feature Engineering Part 2

In [37]:
### Train test split
X = df.drop('Exited', axis=1)
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [38]:
categorical_for_encoding = ['Geography', 'Gender', 'AgeGroup', 'CreditScoreBand']
numeric_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'BalanceToSalaryRatio', 'TenurePerAge']

num_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')) 
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numeric_cols),
        ('cat', cat_transformer, categorical_for_encoding)
    ],
    remainder='passthrough' # Keeps any columns not specified
)

pd.DataFrame(preprocessor.fit_transform(X_train), columns=preprocessor.get_feature_names_out())

,num__CreditScore,num__Age,num__Tenure,num__Balance,num__NumOfProducts,num__EstimatedSalary,num__BalanceToSalaryRatio,num__TenurePerAge,cat__Geography_Germany,cat__Geography_Spain,...,cat__AgeGroup_56-65,cat__CreditScoreBand_Fair,cat__CreditScoreBand_Good,cat__CreditScoreBand_Low,cat__CreditScoreBand_Very Good,remainder__HasCrCard,remainder__IsActiveMember,remainder__IsHighBalance,remainder__IsMultiProductCustomer,remainder__ActiveMember_And_MultiProduct
0,0.322128,-0.526650,1.020441,-0.025128,-0.939389,-1.173495,-0.024114,1.102363,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1,0.959401,0.167184,-0.706997,0.689279,-0.939389,-1.289114,0.107262,-0.719038,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
2,-0.377828,2.479965,1.020441,-1.222350,0.874111,1.140939,-0.155224,-0.073352,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0
3,1.001190,1.323575,0.329466,2.018186,-0.939389,-1.443125,0.517051,-0.218009,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0
4,1.074319,-0.295372,-0.361509,-1.222350,0.874111,-1.576274,-0.155224,-0.307849,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7607,-0.053968,1.439214,-0.016022,1.055790,0.874111,0.733700,-0.097733,-0.465068,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
7608,0.447493,-0.989206,-0.016022,0.370154,-0.939389,0.212791,-0.104340,0.331872,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
7609,-0.336039,-0.989206,-0.016022,-1.222350,0.874111,1.641684,-0.155224,0.331872,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
7610,0.698224,-0.295372,1.020441,-1.222350,0.874111,-0.092133,-0.155224,0.949907,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


#### Saving the preprocessor, X_train, X_test, y_train, and y_test as pkl file

In [ ]:
# Save the entire pipeline (preprocessor + model)
import joblib
joblib.dump(preprocessor, 'artifacts/churn_model_pipeline.pkl')

import numpy as np

# Save X_train, X_test, y_train, y_test into one compressed file
np.savez_compressed('artifacts/dataset_splits.npz', 
                    X_train=X_train, X_test=X_test, 
                    y_train=y_train, y_test=y_test,
                    columns=X_train.columns.to_list())

# To load it back later:
# loaded = np.load('artifacts/dataset_splits.npz', allow_pickle=True)
# X_train = loaded['X_train']


In [40]:
# newpreprocessor=joblib.load('artifacts/churn_model_pipeline.pkl')
# newpreprocessor.fit_transform(X_train)